# Tutorial: Creating an AnnData Object from Tahoe-100M Dataset
This notebook is intented for users who are familiar with the anndata format for single-cell data. We'll walk through how to parse records in the huggingface dataset format and convert between the two.

## Install Required Libraries

## Import Libraries

In [1]:
from datasets import load_dataset
from scipy.sparse import csr_matrix
import anndata
import pandas as pd
import pubchempy as pcp

/uufs/chpc.utah.edu/common/HIPAA/u1521317/software/pkg/miniforge3/envs/Tahoe100M_Study/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Mapping records to anndata

This function takes in a generator that emits records from the Tahoe-100M huggingface dataset and returns an anndata object. Use the `sample_size` argument to specify the number of records you need. You can also create a new generator using the `dataset.filter` function to only emit records that match a certain filter (eg: for a specific drug/plate/sample).

If you'd like to create a DataLoader for an ML training application, it's likely best to use the data in it's native format without interfacing with anndata.

In [2]:

def create_anndata_from_generator(generator, gene_vocab, sample_size=None):
    sorted_vocab_items = sorted(gene_vocab.items())
    token_ids, gene_names = zip(*sorted_vocab_items)
    token_id_to_col_idx = {token_id: idx for idx, token_id in enumerate(token_ids)}

    data, indices, indptr = [], [], [0]
    obs_data = []

    for i, cell in enumerate(generator):
        if sample_size is not None and i >= sample_size:
            break
        genes = cell['genes']
        expressions = cell['expressions']
        if expressions[0] < 0:
            genes = genes[1:]
            expressions = expressions[1:]

        col_indices = [token_id_to_col_idx[gene] for gene in genes if gene in token_id_to_col_idx]
        valid_expressions = [expr for gene, expr in zip(genes, expressions) if gene in token_id_to_col_idx]

        data.extend(valid_expressions)
        indices.extend(col_indices)
        indptr.append(len(data))

        obs_entry = {k: v for k, v in cell.items() if k not in ['genes', 'expressions']}
        obs_data.append(obs_entry)

    expr_matrix = csr_matrix((data, indices, indptr), shape=(len(indptr) - 1, len(gene_names)))
    obs_df = pd.DataFrame(obs_data)

    adata = anndata.AnnData(X=expr_matrix, obs=obs_df)
    adata.var.index = pd.Index(gene_names, name='ensembl_id')

    return adata


## Load Tahoe-100M Dataset

In [3]:
tahoe_100m_ds = load_dataset('vevotx/Tahoe-100M', streaming=True, split='train')

## Load Gene Metadata

The gene metadata contains the mapping between the integer token IDs used in the dataset and standard identifiers for genes (ensembl IDs and HGNC gene symbols)

In [4]:
gene_metadata = load_dataset("vevotx/Tahoe-100M", name="gene_metadata", split="train")
gene_vocab = {entry["token_id"]: entry["ensembl_id"] for entry in gene_metadata}

## Create AnnData Object

In [5]:
adata = create_anndata_from_generator(tahoe_100m_ds, gene_vocab, sample_size=1000)
adata

/uufs/chpc.utah.edu/common/HIPAA/u1521317/software/pkg/miniforge3/envs/Tahoe100M_Study/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


AnnData object with n_obs × n_vars = 1000 × 62710
    obs: 'drug', 'sample', 'BARCODE_SUB_LIB_ID', 'cell_line_id', 'moa-fine', 'canonical_smiles', 'pubchem_cid', 'plate'

## Inspect Metadata (`adata.obs`)

In [6]:
adata.obs.head()

,drug,sample,BARCODE_SUB_LIB_ID,cell_line_id,moa-fine,canonical_smiles,pubchem_cid,plate
0,8-Hydroxyquinoline,smp_1783,01_001_052-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4
1,8-Hydroxyquinoline,smp_1783,01_001_105-lib_1105,CVCL_0546,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4
2,8-Hydroxyquinoline,smp_1783,01_001_165-lib_1105,CVCL_1717,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4
3,8-Hydroxyquinoline,smp_1783,01_003_094-lib_1105,CVCL_1717,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4
4,8-Hydroxyquinoline,smp_1783,01_003_164-lib_1105,CVCL_1056,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4


## Enrich with Sample Metadata

Although the main data contains several metadata fields, there are some additional columns (such as drug concentration) which are omitted to reduce the size of the data. If they are needed, they may be fetched using the sample_metadata.

In [7]:
sample_metadata = load_dataset("vevotx/Tahoe-100M","sample_metadata", split="train").to_pandas()
adata.obs = pd.merge(adata.obs, sample_metadata.drop(columns=["drug","plate"]), on="sample")
adata.obs.head()

/uufs/chpc.utah.edu/common/HIPAA/u1521317/software/pkg/miniforge3/envs/Tahoe100M_Study/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


,drug,sample,BARCODE_SUB_LIB_ID,cell_line_id,moa-fine,canonical_smiles,pubchem_cid,plate,mean_gene_count,mean_tscp_count,mean_mread_count,mean_pcnt_mito,drugname_drugconc
0,8-Hydroxyquinoline,smp_1783,01_001_052-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]"
1,8-Hydroxyquinoline,smp_1783,01_001_105-lib_1105,CVCL_0546,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]"
2,8-Hydroxyquinoline,smp_1783,01_001_165-lib_1105,CVCL_1717,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]"
3,8-Hydroxyquinoline,smp_1783,01_003_094-lib_1105,CVCL_1717,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]"
4,8-Hydroxyquinoline,smp_1783,01_003_164-lib_1105,CVCL_1056,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]"


## Add Drug Metadata

The drug metadata contains additional information for the compounds used in Tahoe-100M. See the dataset card and our [paper](https://www.biorxiv.org/content/10.1101/2025.02.20.639398v1) for more information about how this information was generated.

In [8]:
drug_metadata = load_dataset("vevotx/Tahoe-100M","drug_metadata", split="train").to_pandas()
adata.obs = pd.merge(adata.obs, drug_metadata.drop(columns=["canonical_smiles","pubchem_cid","moa-fine"]), on="drug")
adata.obs.head()

/uufs/chpc.utah.edu/common/HIPAA/u1521317/software/pkg/miniforge3/envs/Tahoe100M_Study/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


,drug,sample,BARCODE_SUB_LIB_ID,cell_line_id,moa-fine,canonical_smiles,pubchem_cid,plate,mean_gene_count,mean_tscp_count,mean_mread_count,mean_pcnt_mito,drugname_drugconc,targets,moa-broad,human-approved,clinical-trials,gpt-notes-approval
0,8-Hydroxyquinoline,smp_1783,01_001_052-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
1,8-Hydroxyquinoline,smp_1783,01_001_105-lib_1105,CVCL_0546,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
2,8-Hydroxyquinoline,smp_1783,01_001_165-lib_1105,CVCL_1717,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
3,8-Hydroxyquinoline,smp_1783,01_003_094-lib_1105,CVCL_1717,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
4,8-Hydroxyquinoline,smp_1783,01_003_164-lib_1105,CVCL_1056,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."


## Drug Info from PubChem

We also provide the pubchem IDs for the compounds in Tahoe, this can be used to querry additional information as needed.

In [9]:
drug_name = adata.obs["drug"].values[0]
cid = int(float(adata.obs["pubchem_cid"].values[0]))
compound = pcp.Compound.from_cid(cid)

print(f"Name: {drug_name}")
print(f"Synonyms: {compound.synonyms[:10]}")
print(f"Formula: {compound.molecular_formula}")
print(f"SMILES: {compound.isomeric_smiles}")
print(f"Mass: {compound.exact_mass}")

Name: 8-Hydroxyquinoline
Synonyms: ['8-HYDROXYQUINOLINE', 'quinolin-8-ol', '148-24-3', '8-quinolinol', 'Oxyquinoline', 'Oxine', 'Quinophenol', 'Oxychinolin', '8-Quinol', '8-Oxyquinoline']
Formula: C9H7NO
SMILES: None
Mass: 145.052763847


## Load Cell Line Metadata
The cell-line metadata contains additional identifiers for the
cell-lines used in Tahoe (eg: Depmap-IDs) as well as a curated list of driver mutations for each cell line. This information can be used for instance to train genotype aware models on the Tahoe data.

In [10]:
cell_line_metadata = load_dataset("vevotx/Tahoe-100M","cell_line_metadata", split="train").to_pandas()
cell_line_metadata.head()

,cell_name,Cell_ID_DepMap,Cell_ID_Cellosaur,Organ,Driver_Gene_Symbol,Driver_VarZyg,Driver_VarType,Driver_ProtEffect_or_CdnaEffect,Driver_Mech_InferDM,Driver_GeneType_DM
0,A549,ACH-000681,CVCL_0023,Lung,CDKN2A,Hom,Deletion,DEL,LoF,Suppressor
1,A549,ACH-000681,CVCL_0023,Lung,CDKN2B,Hom,Deletion,DEL,LoF,Suppressor
2,A549,ACH-000681,CVCL_0023,Lung,KRAS,Hom,Missense,p.G12S,GoF,Oncogene
3,A549,ACH-000681,CVCL_0023,Lung,SMARCA4,Hom,Frameshift,p.Q729fs,LoF,Suppressor
4,A549,ACH-000681,CVCL_0023,Lung,STK11,Hom,Stopgain,p.Q37*,LoF,Suppressor


In [11]:
tahoe_100m_ds

IterableDataset({
    features: ['genes', 'expressions', 'drug', 'sample', 'BARCODE_SUB_LIB_ID', 'cell_line_id', 'moa-fine', 'canonical_smiles', 'pubchem_cid', 'plate'],
    num_shards: 3388
})

In [12]:
adata.obs

,drug,sample,BARCODE_SUB_LIB_ID,cell_line_id,moa-fine,canonical_smiles,pubchem_cid,plate,mean_gene_count,mean_tscp_count,mean_mread_count,mean_pcnt_mito,drugname_drugconc,targets,moa-broad,human-approved,clinical-trials,gpt-notes-approval
0,8-Hydroxyquinoline,smp_1783,01_001_052-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
1,8-Hydroxyquinoline,smp_1783,01_001_105-lib_1105,CVCL_0546,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
2,8-Hydroxyquinoline,smp_1783,01_001_165-lib_1105,CVCL_1717,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
3,8-Hydroxyquinoline,smp_1783,01_003_094-lib_1105,CVCL_1717,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
4,8-Hydroxyquinoline,smp_1783,01_003_164-lib_1105,CVCL_1056,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,Trifluridine,smp_1785,03_040_156-lib_1105,CVCL_0397,DNA synthesis/repair inhibitor,C1C(C(OC1N2C=C(C(=O)NC2=O)C(F)(F)F)CO)O,6256.0,plate4,1281.333120,1996.038180,2332.914887,0.058380,"[('Trifluridine', 0.05, 'uM')]",None,inhibitor/antagonist,yes,yes,Used to treat viral infections and in cancer t...
996,Trifluridine,smp_1785,03_041_013-lib_1105,CVCL_1478,DNA synthesis/repair inhibitor,C1C(C(OC1N2C=C(C(=O)NC2=O)C(F)(F)F)CO)O,6256.0,plate4,1281.333120,1996.038180,2332.914887,0.058380,"[('Trifluridine', 0.05, 'uM')]",None,inhibitor/antagonist,yes,yes,Used to treat viral infections and in cancer t...
997,Trifluridine,smp_1785,03_041_018-lib_1105,CVCL_0399,DNA synthesis/repair inhibitor,C1C(C(OC1N2C=C(C(=O)NC2=O)C(F)(F)F)CO)O,6256.0,plate4,1281.333120,1996.038180,2332.914887,0.058380,"[('Trifluridine', 0.05, 'uM')]",None,inhibitor/antagonist,yes,yes,Used to treat viral infections and in cancer t...
998,Trifluridine,smp_1785,03_041_166-lib_1105,CVCL_0359,DNA synthesis/repair inhibitor,C1C(C(OC1N2C=C(C(=O)NC2=O)C(F)(F)F)CO)O,6256.0,plate4,1281.333120,1996.038180,2332.914887,0.058380,"[('Trifluridine', 0.05, 'uM')]",None,inhibitor/antagonist,yes,yes,Used to treat viral infections and in cancer t...


In [13]:
adata.obs['cell_line_id']

0      CVCL_0480
1      CVCL_0546
2      CVCL_1717
3      CVCL_1717
4      CVCL_1056
         ...    
995    CVCL_0397
996    CVCL_1478
997    CVCL_0399
998    CVCL_0359
999    CVCL_0023
Name: cell_line_id, Length: 1000, dtype: object

In [14]:
subset =adata[adata.obs['cell_line_id'] == "CVCL_0480"]

In [15]:
subset.X[0]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1878 stored elements and shape (1, 62710)>

In [16]:
from scipy.sparse import csr_matrix

# Get the sparse row for the ith cell
sparse_row = subset.X[0]

# If it's a 2D sparse matrix (1, n_genes), convert to 1D
if isinstance(sparse_row, csr_matrix):
    sparse_row = sparse_row.tocsr()

# Extract non-zero indices and values
nonzero_indices = sparse_row.indices
nonzero_values = sparse_row.data

# Get gene names (var_names) for those indices
nonzero_genes = subset.var_names[nonzero_indices]

# Optionally zip into a dict or list
nonzero_expression = dict(zip(nonzero_genes, nonzero_values))


In [17]:
dense_array = subset.X[0].toarray().ravel()  # shape (62710,)


In [18]:
from anndata import AnnData

# Dictionary of AnnData objects, one per sample
samples_dict = {
    sample_id: subset[subset.obs['sample'] == sample_id].copy()
    for sample_id in subset.obs['sample'].unique()
}

import anndata as ad

# Combine all per-sample AnnData objects into one
pooled_adata = ad.concat(samples_dict.values(), axis=0, join='outer', merge='same')

pooled_adata.obs

,drug,sample,BARCODE_SUB_LIB_ID,cell_line_id,moa-fine,canonical_smiles,pubchem_cid,plate,mean_gene_count,mean_tscp_count,mean_mread_count,mean_pcnt_mito,drugname_drugconc,targets,moa-broad,human-approved,clinical-trials,gpt-notes-approval
0,8-Hydroxyquinoline,smp_1783,01_001_052-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
24,8-Hydroxyquinoline,smp_1783,01_011_106-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
44,8-Hydroxyquinoline,smp_1783,01_023_046-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
86,8-Hydroxyquinoline,smp_1783,01_044_070-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
103,8-Hydroxyquinoline,smp_1783,01_054_012-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
150,8-Hydroxyquinoline,smp_1783,01_080_128-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
158,8-Hydroxyquinoline,smp_1783,01_082_019-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
167,8-Hydroxyquinoline,smp_1783,01_085_016-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
173,8-Hydroxyquinoline,smp_1783,01_087_082-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
181,8-Hydroxyquinoline,smp_1783,01_090_065-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."


In [19]:
groups = subset.obs.groupby('sample').indices
subset_concat = anndata.concat([subset[inds] for inds in groups.values()], merge="same")
#assert np.array_equal(subset.X, subset_concat.X)

In [20]:
subset_concat.obs

,drug,sample,BARCODE_SUB_LIB_ID,cell_line_id,moa-fine,canonical_smiles,pubchem_cid,plate,mean_gene_count,mean_tscp_count,mean_mread_count,mean_pcnt_mito,drugname_drugconc,targets,moa-broad,human-approved,clinical-trials,gpt-notes-approval
0,8-Hydroxyquinoline,smp_1783,01_001_052-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
24,8-Hydroxyquinoline,smp_1783,01_011_106-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
44,8-Hydroxyquinoline,smp_1783,01_023_046-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
86,8-Hydroxyquinoline,smp_1783,01_044_070-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
103,8-Hydroxyquinoline,smp_1783,01_054_012-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
150,8-Hydroxyquinoline,smp_1783,01_080_128-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
158,8-Hydroxyquinoline,smp_1783,01_082_019-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
167,8-Hydroxyquinoline,smp_1783,01_085_016-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
173,8-Hydroxyquinoline,smp_1783,01_087_082-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
181,8-Hydroxyquinoline,smp_1783,01_090_065-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."


In [21]:
import anndata as ad

# Step 1: Group by 'sample'
sample_groups = adata.obs.groupby("sample").groups

# Step 2: Make a new AnnData for each sample
adatas_by_sample = {
    sample: adata[sample_groups[sample]].copy() for sample in sample_groups
}

# Step 3: Concatenate them back (optional: keep batch key or ignore)
adata_concat = ad.concat(adatas_by_sample.values(), 
                         join="outer",  # or "inner" depending on your needs
                         label="sample", 
                         keys=adatas_by_sample.keys(), 
                         index_unique=None)


In [22]:
adata_concat.obs

,drug,sample,BARCODE_SUB_LIB_ID,cell_line_id,moa-fine,canonical_smiles,pubchem_cid,plate,mean_gene_count,mean_tscp_count,mean_mread_count,mean_pcnt_mito,drugname_drugconc,targets,moa-broad,human-approved,clinical-trials,gpt-notes-approval
0,8-Hydroxyquinoline,smp_1783,01_001_052-lib_1105,CVCL_0480,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
1,8-Hydroxyquinoline,smp_1783,01_001_105-lib_1105,CVCL_0546,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
2,8-Hydroxyquinoline,smp_1783,01_001_165-lib_1105,CVCL_1717,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
3,8-Hydroxyquinoline,smp_1783,01_003_094-lib_1105,CVCL_1717,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
4,8-Hydroxyquinoline,smp_1783,01_003_164-lib_1105,CVCL_1056,unclear,C1=CC2=C(C(=C1)O)N=CC=C2,1923.0,plate4,1478.268171,2341.339094,2738.463797,0.023783,"[('8-Hydroxyquinoline', 0.05, 'uM')]",None,unclear,no,yes,"Used in some clinical trial formulations, not ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,Trifluridine,smp_1785,03_040_156-lib_1105,CVCL_0397,DNA synthesis/repair inhibitor,C1C(C(OC1N2C=C(C(=O)NC2=O)C(F)(F)F)CO)O,6256.0,plate4,1281.333120,1996.038180,2332.914887,0.058380,"[('Trifluridine', 0.05, 'uM')]",None,inhibitor/antagonist,yes,yes,Used to treat viral infections and in cancer t...
996,Trifluridine,smp_1785,03_041_013-lib_1105,CVCL_1478,DNA synthesis/repair inhibitor,C1C(C(OC1N2C=C(C(=O)NC2=O)C(F)(F)F)CO)O,6256.0,plate4,1281.333120,1996.038180,2332.914887,0.058380,"[('Trifluridine', 0.05, 'uM')]",None,inhibitor/antagonist,yes,yes,Used to treat viral infections and in cancer t...
997,Trifluridine,smp_1785,03_041_018-lib_1105,CVCL_0399,DNA synthesis/repair inhibitor,C1C(C(OC1N2C=C(C(=O)NC2=O)C(F)(F)F)CO)O,6256.0,plate4,1281.333120,1996.038180,2332.914887,0.058380,"[('Trifluridine', 0.05, 'uM')]",None,inhibitor/antagonist,yes,yes,Used to treat viral infections and in cancer t...
998,Trifluridine,smp_1785,03_041_166-lib_1105,CVCL_0359,DNA synthesis/repair inhibitor,C1C(C(OC1N2C=C(C(=O)NC2=O)C(F)(F)F)CO)O,6256.0,plate4,1281.333120,1996.038180,2332.914887,0.058380,"[('Trifluridine', 0.05, 'uM')]",None,inhibitor/antagonist,yes,yes,Used to treat viral infections and in cancer t...


In [23]:
import scanpy as sc
adatas = []
for group, idx in subset.obs.groupby("sample").indices.items():
    sub_adata = subset[idx].copy()
    sub_adata.obsm["qc"], sub_adata.varm[f"{group}_qc"] = sc.pp.calculate_qc_metrics(
        sub_adata, percent_top=(), inplace=False, log1p=False
    )
    adatas.append(sub_adata)

In [24]:
ad.concat(adatas, merge="same", label="sample", keys=[adata.obs['sample'].unique()[i] for i in range(len(adatas))])

AnnData object with n_obs × n_vars = 45 × 62710
    obs: 'drug', 'sample', 'BARCODE_SUB_LIB_ID', 'cell_line_id', 'moa-fine', 'canonical_smiles', 'pubchem_cid', 'plate', 'mean_gene_count', 'mean_tscp_count', 'mean_mread_count', 'mean_pcnt_mito', 'drugname_drugconc', 'targets', 'moa-broad', 'human-approved', 'clinical-trials', 'gpt-notes-approval'
    obsm: 'qc'

In [25]:
adatas

[AnnData object with n_obs × n_vars = 21 × 62710
     obs: 'drug', 'sample', 'BARCODE_SUB_LIB_ID', 'cell_line_id', 'moa-fine', 'canonical_smiles', 'pubchem_cid', 'plate', 'mean_gene_count', 'mean_tscp_count', 'mean_mread_count', 'mean_pcnt_mito', 'drugname_drugconc', 'targets', 'moa-broad', 'human-approved', 'clinical-trials', 'gpt-notes-approval'
     obsm: 'qc'
     varm: 'smp_1783_qc',
 AnnData object with n_obs × n_vars = 16 × 62710
     obs: 'drug', 'sample', 'BARCODE_SUB_LIB_ID', 'cell_line_id', 'moa-fine', 'canonical_smiles', 'pubchem_cid', 'plate', 'mean_gene_count', 'mean_tscp_count', 'mean_mread_count', 'mean_pcnt_mito', 'drugname_drugconc', 'targets', 'moa-broad', 'human-approved', 'clinical-trials', 'gpt-notes-approval'
     obsm: 'qc'
     varm: 'smp_1784_qc',
 AnnData object with n_obs × n_vars = 8 × 62710
     obs: 'drug', 'sample', 'BARCODE_SUB_LIB_ID', 'cell_line_id', 'moa-fine', 'canonical_smiles', 'pubchem_cid', 'plate', 'mean_gene_count', 'mean_tscp_count', 'mean_mr

In [30]:
import anndata as ad

# Get the unique sample IDs
sample_ids = subset.obs["sample"].unique()

# Group the cells by sample_id and pool them
grouped_adatas = [subset[subset.obs["sample"] == sid].copy() for sid in sample_ids]

# Now you have a list of 3 AnnData objects (if 3 unique sample_ids)
# Optionally, concatenate each group if you want all 3 in one object with a 'group' label:
adata_pooled = ad.concat(grouped_adatas, label="sample", keys=sample_ids)


In [32]:
grouped_adatas = [subset[subset.obs["sample"] == sid].copy() for sid in sample_ids]


In [35]:
grouped_adatas

[AnnData object with n_obs × n_vars = 21 × 62710
     obs: 'drug', 'sample', 'BARCODE_SUB_LIB_ID', 'cell_line_id', 'moa-fine', 'canonical_smiles', 'pubchem_cid', 'plate', 'mean_gene_count', 'mean_tscp_count', 'mean_mread_count', 'mean_pcnt_mito', 'drugname_drugconc', 'targets', 'moa-broad', 'human-approved', 'clinical-trials', 'gpt-notes-approval',
 AnnData object with n_obs × n_vars = 16 × 62710
     obs: 'drug', 'sample', 'BARCODE_SUB_LIB_ID', 'cell_line_id', 'moa-fine', 'canonical_smiles', 'pubchem_cid', 'plate', 'mean_gene_count', 'mean_tscp_count', 'mean_mread_count', 'mean_pcnt_mito', 'drugname_drugconc', 'targets', 'moa-broad', 'human-approved', 'clinical-trials', 'gpt-notes-approval',
 AnnData object with n_obs × n_vars = 8 × 62710
     obs: 'drug', 'sample', 'BARCODE_SUB_LIB_ID', 'cell_line_id', 'moa-fine', 'canonical_smiles', 'pubchem_cid', 'plate', 'mean_gene_count', 'mean_tscp_count', 'mean_mread_count', 'mean_pcnt_mito', 'drugname_drugconc', 'targets', 'moa-broad', 'human-